[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/19_gelu_interview.ipynb)

# 🟢 Solution: GELU Activation（面试版）

## 📋 题目描述

**难度：** Easy

实现 GELU 激活函数。

GELU（高斯误差线性单元）根据输入值平滑地进行门控，广泛用于 BERT 和 GPT 等 Transformer。

**签名:** `my_gelu(x) -> Tensor`

**参数:**
- `x` — 任意形状的输入张量

**返回:** 逐元素 GELU 激活，形状与输入相同

**约束:**
- 精确公式：`x * 0.5 * (1 + erf(x / sqrt(2)))`
- 必须与 `F.gelu` 误差在 1e-4 以内
- `gelu(0) = 0`

**提示：** 精确版：`0.5 * x * (1 + torch.erf(x / sqrt(2)))`
近似版：`0.5*x*(1+tanh(sqrt(2/π)*(x+0.044715*x³)))`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

import math

def my_gelu(x):
    # ---- Step 1: GELU 精确公式 ----
    # GELU(x) = x * P(X ≤ x)，其中 X ~ N(0,1)
    # P(X ≤ x) = Φ(x) = 0.5 * (1 + erf(x / sqrt(2)))
    #
    # erf 是什么？误差函数，衡量正态分布在 [-x, x] 区间的概率
    # torch.erf 是 PyTorch 内置的精确误差函数
    return 0.5 * x * (1.0 + torch.erf(x / math.sqrt(2.0)))

# 面试可能要求写近似版（不用 erf）：
def my_gelu_approx(x):
    # ---- 近似公式（tanh 版本）----
    # GELU(x) ≈ 0.5 * x * (1 + tanh(sqrt(2/π) * (x + 0.044715 * x³)))
    # 来源：用 tanh 函数逼近 erf 函数
    # 精度：与精确版误差 < 1e-4，但计算更快（tanh 比 erf 常数小）
    # GPT-2 和 BERT 常用这个近似版
    return 0.5 * x * (1.0 + torch.tanh(
        math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3))
    ))

# 面试常见追问：
# Q: GELU vs ReLU vs SiLU？
# A: ReLU: max(0,x)，硬门控，0点不可导
#    GELU: x*Φ(x)，基于正态CDF的软门控
#    SiLU/Swish: x*sigmoid(x)，基于sigmoid的软门控
#    三者在正区间都接近恒等，负区间行为不同
# Q: 为什么 Transformer 用 GELU 而不是 ReLU？
# A: GELU 平滑可导，梯度更连续，训练更稳定。实验效果也略好。


In [ ]:
x = torch.tensor([-2., -1., 0., 1., 2.])
print('GELU:', my_gelu(x))
print('F.gelu:', F.gelu(x))


In [ ]:
from torch_judge import check
check('gelu')


## 📝 核心思路总结

1. **GELU = x × Φ(x)**：用标准正态 CDF 作为门控信号
2. **erf 函数**：`Φ(x) = 0.5*(1 + erf(x/√2))`，PyTorch 内置 `torch.erf`
3. **软门控**：相比 ReLU 的硬门控，GELU 在零点附近平滑过渡
4. **近似版**：tanh 近似精度足够，计算更快
